# Pure water/steam cooling and condensation inside tubes (v0.6.2)

Cooling and condensing **pure H2O** (no carrier gas) flowing inside bare tubes: a
steam heater / desuperheater-condenser-subcooler. Unlike v0.6.1's wet-gas
condensation model (dew point, humidity ratio `W`, Chilton-Colburn/Lewis mass
transfer), this is a direct vapor-liquid phase-equilibrium problem parameterized by
vapor quality `x`, solved with `IAPWS97WaterSteamProvider` on the inside.

Four cases:

- **A.** Saturated vapor inlet -> partial condensation.
- **B.** Wet steam inlet (`x_in < 1`) -> further condensation to a lower `x_out`.
- **C.** Superheated steam inlet -> desuperheating + condensation.
- **D.** Superheated steam inlet, large surface -> complete condensation +
  condensate subcooling.

Each case reports: `phase_in`/`phase_out`, `T_in`/`T_out`, `T_sat`, `h_in`/`h_out`,
`quality_in`/`quality_out`, `Q_desuperheat`/`Q_condensation`/`Q_subcooling`/`Q_total`,
area fractions, zone heat-transfer coefficients, warnings, and the two-phase
pressure-drop support status.


In [1]:
import sys
from pathlib import Path

repository_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'core').is_dir())
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from core.geometry.bundle import TubeBundle
from core.geometry.tube import BareTube
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.properties.gas_mixture import GasMixturePropertyProvider, GasMixtureSpec
from core.properties.water import IAPWS97WaterSteamProvider

P = 101_325.0  # Pa


def make_hx(n_rows, n_tubes_per_row, length_total, flow_arrangement='crossflow'):
    tube = BareTube(D_i=0.020, D_o=0.025, length_total=length_total, length_effective=length_total, wall_k=50.0)
    bundle = TubeBundle(
        tube=tube, n_rows=n_rows, n_tubes_per_row=n_tubes_per_row,
        pitch_transverse=0.040, pitch_longitudinal=0.040,
        layout='staggered', n_passes_tube=1, flow_arrangement=flow_arrangement,
    )
    return BareTubeHeatExchanger(bundle)


def dry_air_provider():
    return GasMixturePropertyProvider(GasMixtureSpec(components={'N2': 0.79, 'O2': 0.21}, basis='mole'))


def summarize(label, result):
    pc = result.inside_phase_change
    print(f"=== {label} ===")
    print(f"phase_in={pc.phase_in.value:<18s} phase_out={pc.phase_out.value}")
    print(f"T_in={pc.T_in:8.3f} K   T_out={pc.T_out:8.3f} K   T_sat={pc.T_sat:8.3f} K")
    print(f"h_in={pc.h_in:12.1f} J/kg   h_out={pc.h_out:12.1f} J/kg")
    print(f"quality_in={pc.quality_in}   quality_out={pc.quality_out}")
    print(f"Q_desuperheat={pc.Q_desuperheat:10.1f} W   Q_condensation={pc.Q_condensation:10.1f} W   Q_subcooling={pc.Q_subcooling:10.1f} W   Q_total={pc.Q_total:10.1f} W")
    print(f"f_desuperheat={pc.f_desuperheat:.4f}   f_condensation={pc.f_condensation:.4f}   f_subcooling={pc.f_subcooling:.4f}")
    print(f"zone alpha [W/(m2*K)]: desuperheat={pc.zone_alpha_desuperheat}, condensation={pc.zone_alpha_condensation}, subcooling={pc.zone_alpha_subcooling}")
    print(f"two_phase_pressure_drop_supported={pc.two_phase_pressure_drop_supported}")
    print('warnings:')
    for w in pc.warnings:
        print(f"  [{w.severity}] {w.code}: {w.message}")
    print()
    return pc


## Case A -- saturated vapor inlet -> partial condensation

INPUT: saturated steam (`x_in=1.0`) at 1 atm, `m_dot=0.15 kg/s`, cooled by dry air at 290 K.

In [2]:
hx_a = make_hx(n_rows=4, n_tubes_per_row=6, length_total=6.0)
inside_a = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.15, x_in=1.0, p=P)
outside_a = HXSideInput(provider=dry_air_provider(), m_dot=2.0, T_in=290.0, p=P)

result_a = hx_a.simulate(inside_a, outside_a)
pc_a = summarize('A. Saturated vapor -> partial condensation', result_a)
assert pc_a.active and 0.0 < pc_a.quality_out < 1.0


=== A. Saturated vapor -> partial condensation ===
phase_in=saturated_vapor    phase_out=two_phase
T_in= 373.124 K   T_out= 373.124 K   T_sat= 373.124 K
h_in=   2675531.5 J/kg   h_out=   2500821.3 J/kg
quality_in=1.0   quality_out=0.9225760931149125
Q_desuperheat=       0.0 W   Q_condensation=   26206.5 W   Q_subcooling=       0.0 W   Q_total=   26206.5 W
f_desuperheat=0.0000   f_condensation=1.0000   f_subcooling=0.0000
zone alpha [W/(m2*K)]: desuperheat=None, condensation=8017.275337110288, subcooling=None
two_phase_pressure_drop_supported=False
warnings:
  [info] TWO_PHASE_PRESSURE_DROP_NOT_SUPPORTED: inside: two-phase condensation-zone pressure drop is not modelled in v0.6.2; single-phase tube-side dp components do not represent the full condensing-zone dp.
  [info] INSIDE_CONDENSATION_DETECTED: inside: pure water/steam multi-zone cooling was solved (condensation).



## Case B -- wet steam inlet -> further condensation

INPUT: wet steam at `x_in=0.7` (already partially condensed upstream) at 1 atm, `m_dot=0.15 kg/s`,
cooled by dry air at 290 K.

In [3]:
hx_b = make_hx(n_rows=4, n_tubes_per_row=6, length_total=6.0)
inside_b = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.15, x_in=0.7, p=P)
outside_b = HXSideInput(provider=dry_air_provider(), m_dot=2.0, T_in=290.0, p=P)

result_b = hx_b.simulate(inside_b, outside_b)
pc_b = summarize('B. Wet steam inlet -> lower x_out', result_b)
assert pc_b.quality_out < pc_b.quality_in


=== B. Wet steam inlet -> lower x_out ===
phase_in=two_phase          phase_out=two_phase
T_in= 373.124 K   T_out= 373.124 K   T_sat= 373.124 K
h_in=   1998569.2 J/kg   h_out=   1823998.9 J/kg
quality_in=0.7   quality_out=0.6226380740292368
Q_desuperheat=       0.0 W   Q_condensation=   26185.6 W   Q_subcooling=       0.0 W   Q_total=   26185.6 W
f_desuperheat=0.0000   f_condensation=1.0000   f_subcooling=0.0000
zone alpha [W/(m2*K)]: desuperheat=None, condensation=6770.696463391846, subcooling=None
two_phase_pressure_drop_supported=False
warnings:
  [info] TWO_PHASE_PRESSURE_DROP_NOT_SUPPORTED: inside: two-phase condensation-zone pressure drop is not modelled in v0.6.2; single-phase tube-side dp components do not represent the full condensing-zone dp.
  [info] INSIDE_CONDENSATION_DETECTED: inside: pure water/steam multi-zone cooling was solved (condensation).



## Case C -- superheated steam inlet -> desuperheating + condensation

INPUT: superheated steam at 450 K, 1 atm, `m_dot=0.15 kg/s`, cooled by dry air at 290 K.

In [4]:
hx_c = make_hx(n_rows=4, n_tubes_per_row=6, length_total=6.0)
inside_c = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.15, T_in=450.0, p=P)
outside_c = HXSideInput(provider=dry_air_provider(), m_dot=2.0, T_in=290.0, p=P)

result_c = hx_c.simulate(inside_c, outside_c)
pc_c = summarize('C. Superheated -> desuperheating + condensation', result_c)
assert pc_c.Q_desuperheat > 0.0 and pc_c.Q_condensation > 0.0


=== C. Superheated -> desuperheating + condensation ===
phase_in=superheated_vapor  phase_out=two_phase
T_in= 450.000 K   T_out= 373.124 K   T_sat= 373.124 K
h_in=   2829672.0 J/kg   h_out=   2641978.5 J/kg
quality_in=None   quality_out=0.9851307896897197
Q_desuperheat=   23121.1 W   Q_condensation=    5032.9 W   Q_subcooling=       0.0 W   Q_total=   28154.0 W
f_desuperheat=0.8072   f_condensation=0.1928   f_subcooling=0.0000
zone alpha [W/(m2*K)]: desuperheat=115.05265585022195, condensation=7678.024823936838, subcooling=None
two_phase_pressure_drop_supported=False
warnings:
  [info] TWO_PHASE_PRESSURE_DROP_NOT_SUPPORTED: inside: two-phase condensation-zone pressure drop is not modelled in v0.6.2; single-phase tube-side dp components do not represent the full condensing-zone dp.
  [info] INSIDE_CONDENSATION_DETECTED: inside: pure water/steam multi-zone cooling was solved (desuperheat, condensation).



## Case D -- superheated steam, large surface -> complete condensation + subcooling

INPUT: superheated steam at 450 K, 1 atm, `m_dot=0.4 kg/s`, a larger exchanger and colder outside
air (280 K) so the exchanger fully condenses the steam and subcools the condensate.

In [5]:
hx_d = make_hx(n_rows=16, n_tubes_per_row=16, length_total=20.0)
inside_d = HXSideInput(provider=IAPWS97WaterSteamProvider(), m_dot=0.4, T_in=450.0, p=P)
outside_d = HXSideInput(provider=dry_air_provider(), m_dot=50.0, T_in=280.0, p=P)

result_d = hx_d.simulate(inside_d, outside_d)
pc_d = summarize('D. Superheated -> complete condensation -> subcooled condensate', result_d)
assert pc_d.Q_desuperheat > 0.0 and pc_d.Q_condensation > 0.0 and pc_d.Q_subcooling > 0.0
assert pc_d.quality_out is None  # subcooled liquid


=== D. Superheated -> complete condensation -> subcooled condensate ===
phase_in=superheated_vapor  phase_out=subcooled_liquid
T_in= 450.000 K   T_out= 291.743 K   T_sat= 373.124 K
h_in=   2829672.0 J/kg   h_out=     78123.0 J/kg
quality_in=None   quality_out=None
Q_desuperheat=   61656.2 W   Q_condensation=  902616.3 W   Q_subcooling=  136347.1 W   Q_total= 1100619.6 W
f_desuperheat=0.0645   f_condensation=0.4475   f_subcooling=0.4880
zone alpha [W/(m2*K)]: desuperheat=37.24736856746274, condensation=1745.121195973727, subcooling=123.92890715242872
two_phase_pressure_drop_supported=False
warnings:
  [warning] SHAH_CONDENSATION_OUT_OF_RANGE: G = 4.97359 kg/(m2*s) is outside the Shah (1979) correlation's documented applicability range [10.8, 210.6] kg/(m2*s); this result is an extrapolation.
  [info] TWO_PHASE_PRESSURE_DROP_NOT_SUPPORTED: inside: two-phase condensation-zone pressure drop is not modelled in v0.6.2; single-phase tube-side dp components do not represent the full condensing

## Rating: closing a known heat balance (Case D)

Reuses the same steam-side inlet/outlet resolution and multi-zone physics as
Simulation -- ``Rating`` is only the "given both temperature programs, find the
required area / overdesign factor" orchestration on top.

In [6]:
from core.models.heat_balance import BalanceSideSpec

# Independent, comfortably-achievable closed balance (counterflow, so the
# maximum-achievable-effectiveness ceiling is not a binding constraint here).
hx_rating = make_hx(n_rows=16, n_tubes_per_row=16, length_total=20.0, flow_arrangement='counterflow')

rating_d = hx_rating.rate(
    BalanceSideSpec(provider=IAPWS97WaterSteamProvider(), p=P, m_dot=0.4, T_in=450.0, T_out=300.0),
    BalanceSideSpec(provider=dry_air_provider(), p=P, m_dot=50.0, T_in=280.0, T_out=300.0),
)
rpc_d = rating_d.inside_phase_change
print(f"overdesign_factor={rating_d.overdesign_factor:.4f}")
print(f"A_required (outside-area basis) [m2]={rating_d.A_required:.4f}")
print(f"Q_required [W]={rpc_d.Q_total:.1f}")
print(f"phase_out={rpc_d.phase_out.value}")


overdesign_factor=0.3564
A_required (outside-area basis) [m2]=296.4713
Q_required [W]=1086802.8
phase_out=subcooled_liquid
